In [1]:
import sqlite3
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

DB_PATH = "../db/fpl.db"
con = sqlite3.connect(DB_PATH)

print("Connected.")

Connected.


In [2]:
df = pd.read_sql("""
    SELECT
        pgs.player_id,
        pgs.gameweek_id,
        pgs.season,
        p.web_name,
        pos.singular_name_short         AS position,
        pgs.was_home,
        pgs.opponent_team,
        pgs.value,
        pgs.minutes,
        pgs.starts,
        pgs.total_points,
        pgs.goals_scored,
        pgs.assists,
        pgs.clean_sheets,
        pgs.expected_goals,
        pgs.expected_assists,
        pgs.expected_goal_involvements,
        pgs.bonus,
        pgs.bps,
        pgs.saves
    FROM player_gameweek_stats pgs
    JOIN players p ON pgs.player_id = p.id
    JOIN positions pos ON p.position_id = pos.id
    ORDER BY pgs.player_id, pgs.season, pgs.gameweek_id
""", con)

print(f"Rows: {len(df):,}")
print(f"Seasons: {df['season'].unique()}")
print(f"Players: {df['player_id'].nunique():,}")
df.head(10)

Rows: 104,383
Seasons: <StringArray>
['2022-23', '2023-24', '2024-25', '2025-26']
Length: 4, dtype: str
Players: 825


,player_id,gameweek_id,season,web_name,position,was_home,opponent_team,value,minutes,starts,total_points,goals_scored,assists,clean_sheets,expected_goals,expected_assists,expected_goal_involvements,bonus,bps,saves
0,1,1,2022-23,Raya,GKP,0,7,45,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0
1,1,2,2022-23,Raya,GKP,1,10,44,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0
2,1,3,2022-23,Raya,GKP,0,3,44,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0
3,1,4,2022-23,Raya,GKP,1,9,43,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0
4,1,5,2022-23,Raya,GKP,1,2,43,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0
5,1,6,2022-23,Raya,GKP,0,14,42,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0
6,1,8,2022-23,Raya,GKP,0,4,42,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0
7,1,9,2022-23,Raya,GKP,1,18,42,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0
8,1,10,2022-23,Raya,GKP,1,12,42,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0
9,1,11,2022-23,Raya,GKP,0,11,42,0,0,0,0,0,0,0.0,0.0,0.0,0,0,0


we want to target next week's points

In [3]:
df['target_next_gw_points'] = (
    df.groupby(['player_id', 'season'])['total_points']
    .shift(-1)
)

print(f"Rows with a valid target: {df['target_next_gw_points'].notna().sum():,}")
print(f"Rows without a target (last GW of each season): {df['target_next_gw_points'].isna().sum():,}")
df[['web_name', 'season', 'gameweek_id', 'total_points', 'target_next_gw_points']].head(10)


Rows with a valid target: 101,152
Rows without a target (last GW of each season): 3,231


,web_name,season,gameweek_id,total_points,target_next_gw_points
0,Raya,2022-23,1,0,0.0
1,Raya,2022-23,2,0,0.0
2,Raya,2022-23,3,0,0.0
3,Raya,2022-23,4,0,0.0
4,Raya,2022-23,5,0,0.0
5,Raya,2022-23,6,0,0.0
6,Raya,2022-23,8,0,0.0
7,Raya,2022-23,9,0,0.0
8,Raya,2022-23,10,0,0.0
9,Raya,2022-23,11,0,0.0


compute the rolling form, ie. average points last 3 and 5 game weeks

In [4]:
df['form_3gw'] = (
    df.groupby(['player_id', 'season'])['total_points']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

df['form_5gw'] = (
    df.groupby(['player_id', 'season'])['total_points']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# Verify on a known player
salah = df[(df['web_name'] == 'M.Salah') & (df['season'] == '2025-26')][
    ['gameweek_id', 'total_points', 'form_3gw', 'form_5gw']
].head(10)

print(salah.to_string())

       gameweek_id  total_points  form_3gw  form_5gw
53522            1             8       NaN       NaN
53523            2             5  8.000000  8.000000
53524            3             3  6.500000  6.500000
53525            4             9  5.333333  5.333333
53526            5             5  5.666667  6.250000
53527            6             2  5.666667  6.000000
53528            7             2  5.333333  4.800000
53529            8             2  3.000000  4.200000
53530            9             6  2.000000  4.000000
53531           10            10  3.333333  3.400000


Minute adjusted stats

In [5]:
df['xg_per_90'] = np.where(
    df['minutes'] > 0,
    (df['expected_goals'] / df['minutes']) * 90,
    0
)

df['xa_per_90'] = np.where(
    df['minutes'] > 0,
    (df['expected_assists'] / df['minutes']) * 90,
    0
)

df['xgi_per_90'] = np.where(
    df['minutes'] > 0,
    (df['expected_goal_involvements'] / df['minutes']) * 90,
    0
)

# Verify — find a gameweek where Haaland played and check the maths
haaland = df[(df['web_name'] == 'Haaland') & (df['season'] == '2025-26')][
    ['gameweek_id', 'minutes', 'expected_goals', 'xg_per_90']
].head(8)

print(haaland.to_string())


       gameweek_id  minutes  expected_goals  xg_per_90
60468            1       72            1.99   2.487500
60469            2       90            0.47   0.470000
60470            3       90            1.40   1.400000
60471            4       86            1.80   1.883721
60472            5       75            0.52   0.624000
60473            6       90            1.17   1.170000
60474            7       90            0.29   0.290000
60475            8       90            0.98   0.980000


**important limitation**: for the team difficulty, we'll start by using the static difficulty as of GW31. Later down the line we can go back and find more accurate ways to tie this to past seasons.

In [10]:
# Reload teams with id included
teams = pd.read_sql("""
    SELECT 
        id,
        name,
        strength_overall_home,
        strength_overall_away,
        strength_attack_home,
        strength_attack_away,
        strength_defence_home,
        strength_defence_away
    FROM teams
""", con)

# Name-based lookups for 2025-26 (API stores opponent as team name)
opponent_home_strength = teams.set_index('name')['strength_overall_home'].to_dict()
opponent_away_strength = teams.set_index('name')['strength_overall_away'].to_dict()

# ID-based lookups for historical rows (vaastav stores opponent as integer string)
opponent_home_strength_id = teams.set_index('id')['strength_overall_home'].to_dict()
opponent_away_strength_id = teams.set_index('id')['strength_overall_away'].to_dict()

# Recompute opponent_strength from scratch using both lookup methods
def get_opponent_strength(row):
    try:
        opp_id = int(row['opponent_team'])
        if row['was_home'] == 1:
            return opponent_away_strength_id.get(opp_id)
        else:
            return opponent_home_strength_id.get(opp_id)
    except (ValueError, TypeError):
        # opponent_team is a name string (2025-26) — use name lookup
        if row['was_home'] == 1:
            return opponent_away_strength.get(row['opponent_team'])
        else:
            return opponent_home_strength.get(row['opponent_team'])

df['opponent_strength'] = df.apply(get_opponent_strength, axis=1)

print(f"Rows with opponent_strength: {df['opponent_strength'].notna().sum():,}")
print(f"Rows still missing: {df['opponent_strength'].isna().sum():,}")
print(df[['web_name', 'season', 'gameweek_id', 'was_home', 'opponent_team', 'opponent_strength']].head(10).to_string())


Rows with opponent_strength: 104,383
Rows still missing: 0
  web_name   season  gameweek_id  was_home opponent_team  opponent_strength
0     Raya  2022-23            1         0             7               1205
1     Raya  2022-23            2         1            10               1165
2     Raya  2022-23            3         0             3                975
3     Raya  2022-23            4         1             9               1165
4     Raya  2022-23            5         1             2               1225
5     Raya  2022-23            6         0            14               1180
6     Raya  2022-23            8         0             4               1130
7     Raya  2022-23            9         1            18               1160
8     Raya  2022-23           10         1            12               1245
9     Raya  2022-23           11         0            11               1090


Rolling xG and xA

In [11]:
for col in ['xg_per_90', 'xa_per_90', 'xgi_per_90']:
    df[f'{col}_roll3'] = (
        df.groupby(['player_id', 'season'])[col]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    )
    df[f'{col}_roll5'] = (
        df.groupby(['player_id', 'season'])[col]
        .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    )

# Verify on Haaland
print(df[(df['web_name'] == 'Haaland') & (df['season'] == '2025-26')][
    ['gameweek_id', 'minutes', 'xg_per_90', 'xg_per_90_roll3', 'xg_per_90_roll5']
].head(8).to_string())


       gameweek_id  minutes  xg_per_90  xg_per_90_roll3  xg_per_90_roll5
60468            1       72   2.487500              NaN              NaN
60469            2       90   0.470000         2.487500         2.487500
60470            3       90   1.400000         1.478750         1.478750
60471            4       86   1.883721         1.452500         1.452500
60472            5       75   0.624000         1.251240         1.560305
60473            6       90   1.170000         1.302574         1.373044
60474            7       90   0.290000         1.225907         1.109544
60475            8       90   0.980000         0.694667         1.073544


Cumulative stats

In [12]:
for col in ['goals_scored', 'assists', 'clean_sheets', 'bonus', 'saves']:
    df[f'{col}_cumulative'] = (
        df.groupby(['player_id', 'season'])[col]
        .transform(lambda x: x.shift(1).cumsum())
    )

# Verify on Salah
print(df[(df['web_name'] == 'M.Salah') & (df['season'] == '2025-26')][
    ['gameweek_id', 'goals_scored', 'goals_scored_cumulative',
     'assists', 'assists_cumulative']
].head(10).to_string())


       gameweek_id  goals_scored  goals_scored_cumulative  assists  assists_cumulative
53522            1             1                      NaN        0                 NaN
53523            2             0                      1.0        1                 0.0
53524            3             0                      1.0        0                 1.0
53525            4             1                      1.0        0                 1.0
53526            5             0                      2.0        1                 1.0
53527            6             0                      2.0        0                 2.0
53528            7             0                      2.0        0                 2.0
53529            8             0                      2.0        0                 2.0
53530            9             1                      2.0        0                 2.0
53531           10             1                      3.0        0                 2.0


Flags for if the player played last gameweek

In [ ]:
df['started_last_gw'] = (
    df.groupby(['player_id', 'season'])['starts']
    .transform(lambda x: x.shift(1))
)

df['played_last_gw'] = (
    df.groupby(['player_id', 'season'])['minutes']
    .transform(lambda x: (x.shift(1) > 0).astype(int))
)

# Also carry forward current gameweek price as a feature
print("Features so far:")
feature_cols = [
    'form_3gw', 'form_5gw',
    'xg_per_90_roll3', 'xg_per_90_roll5',
    'xa_per_90_roll3', 'xa_per_90_roll5',
    'xgi_per_90_roll3', 'xgi_per_90_roll5',
    'opponent_strength', 'was_home',
    'goals_scored_cumulative', 'assists_cumulative',
    'clean_sheets_cumulative', 'bonus_cumulative',
    'saves_cumulative', 'started_last_gw', 'played_last_gw',
    'value'
]
print(f"Total features: {len(feature_cols)}")
print(df[feature_cols].notna().sum())


Features so far:
Total features: 18
form_3gw                   101152
form_5gw                   101152
xg_per_90_roll3            101152
xg_per_90_roll5            101152
xa_per_90_roll3            101152
xa_per_90_roll5            101152
xgi_per_90_roll3           101152
xgi_per_90_roll5           101152
opponent_strength          104383
was_home                   104383
goals_scored_cumulative    101152
assists_cumulative         101152
clean_sheets_cumulative    101152
bonus_cumulative           101152
saves_cumulative           101152
started_last_gw            101152
played_last_gw             104383
value                      104383
dtype: int64


Building the feature Matrix

In [14]:
identifier_cols = ['player_id', 'gameweek_id', 'season', 'web_name', 'position']

feature_matrix = df[identifier_cols + feature_cols + ['target_next_gw_points']].copy()

# Drop rows where target is null (last GW of each season)
feature_matrix = feature_matrix.dropna(subset=['target_next_gw_points']).reset_index(drop=True)

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"\nNull counts:")
print(feature_matrix.isnull().sum())
print(f"\nSample:")
print(feature_matrix.head(3).to_string())

Feature matrix shape: (101152, 24)

Null counts:
player_id                     0
gameweek_id                   0
season                        0
web_name                      0
position                      0
form_3gw                   3222
form_5gw                   3222
xg_per_90_roll3            3222
xg_per_90_roll5            3222
xa_per_90_roll3            3222
xa_per_90_roll5            3222
xgi_per_90_roll3           3222
xgi_per_90_roll5           3222
opponent_strength             0
was_home                      0
goals_scored_cumulative    3222
assists_cumulative         3222
clean_sheets_cumulative    3222
bonus_cumulative           3222
saves_cumulative           3222
started_last_gw            3222
played_last_gw                0
value                         0
target_next_gw_points         0
dtype: int64

Sample:
   player_id  gameweek_id   season web_name position  form_3gw  form_5gw  xg_per_90_roll3  xg_per_90_roll5  xa_per_90_roll3  xa_per_90_roll5  xgi_per_90_roll3  x

In [15]:
import os

os.makedirs('../data/features', exist_ok=True)
feature_matrix.to_csv('../data/features/feature_matrix.csv', index=False)

print(f"Saved to data/features/feature_matrix.csv")
print(f"Shape: {feature_matrix.shape}")

Saved to data/features/feature_matrix.csv
Shape: (101152, 24)


In [16]:
con.close()
print("Connection closed.")


Connection closed.
